In [1]:
import scanpy as sc
import os
import numpy as np
import pandas as pd
import scipy.sparse as sp
from geneformer import EmbExtractor
from geneformer import TranscriptomeTokenizer
import time
import torch
from memory_profiler import memory_usage
from datetime import datetime
from functools import wraps


def gene2ensembl(gene_list,mapping_path):
    mapping_df = pd.read_csv(mapping_path,header=0)
    mapping_df_unique = mapping_df.drop_duplicates(subset=['Gene name'], keep='first')
    mapping_dict = dict(zip(mapping_df_unique['Gene name'], mapping_df_unique['ensembl ID']))
    mapped_gene_list = []
    unmapped_genes = []
    for gene in gene_list:
        if gene.startswith('ENS'):
            mapped_gene_list.append(gene)
            continue
        if gene in mapping_dict:
            mapped_gene_list.append(mapping_dict[gene])
        else:
            mapped_gene_list.append(gene)
            unmapped_genes.append(gene)
    if unmapped_genes:
        print(f"Warning: {len(unmapped_genes)} genes not found in mapping file: {unmapped_genes[:10]}{'...' if len(unmapped_genes) > 10 else ''}")
    return mapped_gene_list




def measure_resources(cuda_id: int = 0,task_des:str="task",save_dir:str="/home/cavin/jt/benchmark/experiments/results/run_metric/cluster_with_annotation"):
    """
    测量函数运行时的CPU内存、指定GPU的显存消耗(峰值)和运行时间。
    
    参数:
    cuda_id (int): 需要监控的 GPU 编号，例如传入 0 代表监控 cuda:0
    """
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            
            
            device = None
            if torch.cuda.is_available():
                try:
                    if not torch.cuda.is_initialized():
                        torch.cuda.init()
                    device = torch.device(f"cuda:{cuda_id}")
                    torch.cuda.reset_peak_memory_stats(device)
                except Exception as e:
                    print(f"⚠️ GPU {cuda_id} 初始化监控失败: {e}")

            start_time = time.time()
            
            mem_usage_mb, result = memory_usage(
                (func, args, kwargs), 
                max_usage=True, 
                retval=True
            )
            
            end_time = time.time()
            execution_time = end_time - start_time
            
           
            mem_usage_gb = mem_usage_mb / 1024.0
            
            allocated_gb = 0.0
            cached_gb = 0.0

            
            if device is not None:
                try:
                    allocated_gb = torch.cuda.max_memory_allocated(device) / (1024 ** 3)
                    cached_gb = torch.cuda.max_memory_reserved(device) / (1024 ** 3)
                except Exception as e:
                    print(f"⚠️ GPU {cuda_id} 显存统计失败: {e}")

            timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

            # ===========================
            # 美观的表格式输出
            # ===========================
            report = f"""
============================================================
RESOURCE USAGE REPORT: {func.__name__}
============================================================
Timestamp                   : {timestamp}
Target GPU ID               : {cuda_id}
Execution Time (Minutes)    : {execution_time/60:.4f}
Execution Time (Seconds)    : {execution_time:.2f}
CPU Peak Memory Usage (GB)  : {mem_usage_gb:.4f}
GPU Peak Allocated Mem (GB) : {allocated_gb:.4f}
GPU Peak Cached Mem (GB)    : {cached_gb:.4f}
CUDA Available              : {torch.cuda.is_available()}
============================================================
"""
            print(report)
            save_file_name = "record.csv"
            full_path = os.path.join(save_dir, save_file_name)
            os.makedirs(save_dir, exist_ok=True)
            new_record = {"Timestamp":timestamp,"Task":task_des,
                          "Target GPU ID ":cuda_id,
                          "Execution Time (Minutes)":execution_time/60,
                          "Execution Time (Seconds)":execution_time,
                          "CPU Peak Memory Usage (GB)":mem_usage_gb,
                          "GPU Peak Allocated Mem (GB)":allocated_gb,
                          "GPU Peak Cached Mem (GB)":cached_gb}
            new_data_df = pd.DataFrame([new_record])
            if not os.path.isfile(full_path):
                # 如果文件不存在：直接正常写入（默认 mode='w'），自带表头
                new_data_df.to_csv(full_path, index=False)
                print(f"文件不存在，已创建新文件并写入表头：{full_path}")
            else:
                # 如果文件已存在：使用追加模式 (mode='a')，并且绝对不写表头 (header=False)
                new_data_df.to_csv(full_path, mode='a', header=False, index=False)
                print(f"文件已存在，数据已成功追加至末尾。")
            print(f"资源使用记录已保存到: {full_path}")
            return result
        return wrapper
    return decorator




/home/cavin/anaconda3/envs/geneformer/lib/python3.10/site-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/home/cavin/anaconda3/envs/geneformer/lib/python3.10/site-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


In [2]:
simple_path = "/home/cavin/jt/benchmark/data/human_lung_cancer/SMI_Lung.h5ad"
adata = sc.read_h5ad(simple_path)
adata

AnnData object with n_obs × n_vars = 87589 × 960
    obs: 'cell_type', 'niche', 'patient_sample', 'merge_cell_type', 'Area', 'AspectRatio', 'CenterX_local_px', 'CenterY_local_px', 'CenterX_global_px', 'CenterY_global_px', 'Width', 'Height', 'Mean.MembraneStain', 'Max.MembraneStain', 'Mean.PanCK', 'Max.PanCK', 'Mean.CD45', 'Max.CD45', 'Mean.CD3', 'Max.CD3', 'Mean.DAPI', 'Max.DAPI', 'fov', 'cell_ID', 'n_genes'
    var: 'n_cells'
    obsm: 'spatial'

In [3]:
N_HVG = 2000
sc.pp.highly_variable_genes(adata, n_top_genes=N_HVG, flavor='seurat_v3')
adata = adata[:, adata.var['highly_variable']]
adata

View of AnnData object with n_obs × n_vars = 87589 × 960
    obs: 'cell_type', 'niche', 'patient_sample', 'merge_cell_type', 'Area', 'AspectRatio', 'CenterX_local_px', 'CenterY_local_px', 'CenterX_global_px', 'CenterY_global_px', 'Width', 'Height', 'Mean.MembraneStain', 'Max.MembraneStain', 'Mean.PanCK', 'Max.PanCK', 'Mean.CD45', 'Max.CD45', 'Mean.CD3', 'Max.CD3', 'Mean.DAPI', 'Max.DAPI', 'fov', 'cell_ID', 'n_genes'
    var: 'n_cells', 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm'
    uns: 'hvg'
    obsm: 'spatial'

In [4]:
adata.var_names

Index(['AATK', 'ABL1', 'ABL2', 'ACE', 'ACE2', 'ACKR1', 'ACKR3', 'ACKR4',
       'ACTA2', 'ACTG2',
       ...
       'WNT5B', 'WNT7A', 'WNT7B', 'WNT9A', 'XBP1', 'XCL1', 'XCL2', 'YBX3',
       'YES1', 'ZFP36'],
      dtype='object', length=960)

In [5]:
adata_idlist = gene2ensembl(adata.var_names.to_list(),"/home/cavin/jt/benchmark/data/new_genes_homo_sapiens.csv")
adata.var['ensembl_id'] = pd.DataFrame(adata_idlist, index=adata.var.index)
adata.var["gene_name"] = adata.var.index
adata.var.index = adata_idlist
adata.var_names.name = None
adata.var_names

/tmp/ipykernel_2615919/4069467813.py:2: ImplicitModificationWarning: Trying to modify attribute `.var` of view, initializing view as actual.
  adata.var['ensembl_id'] = pd.DataFrame(adata_idlist, index=adata.var.index)


Index(['ENSG00000181409', 'ENSG00000097007', 'ENSG00000143322',
       'ENSG00000159640', 'ENSG00000130234', 'ENSG00000213088',
       'ENSG00000144476', 'ENSG00000129048', 'ENSG00000107796',
       'ENSG00000163017',
       ...
       'ENSG00000111186', 'ENSG00000154764', 'ENSG00000188064',
       'ENSG00000143816', 'ENSG00000100219', 'ENSG00000143184',
       'ENSG00000143185', 'ENSG00000060138', 'ENSG00000176105',
       'ENSG00000128016'],
      dtype='object', length=960)

In [6]:
adata

AnnData object with n_obs × n_vars = 87589 × 960
    obs: 'cell_type', 'niche', 'patient_sample', 'merge_cell_type', 'Area', 'AspectRatio', 'CenterX_local_px', 'CenterY_local_px', 'CenterX_global_px', 'CenterY_global_px', 'Width', 'Height', 'Mean.MembraneStain', 'Max.MembraneStain', 'Mean.PanCK', 'Max.PanCK', 'Mean.CD45', 'Max.CD45', 'Mean.CD3', 'Max.CD3', 'Mean.DAPI', 'Max.DAPI', 'fov', 'cell_ID', 'n_genes'
    var: 'n_cells', 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm', 'ensembl_id', 'gene_name'
    uns: 'hvg'
    obsm: 'spatial'

In [7]:
if 'n_counts' not in adata.obs.columns:
    if 'UMI count' in adata.obs.columns:
        adata.obs['n_counts'] = adata.obs['UMI count']
    else:
        # 简单的 sum 作为 n_counts
        adata.obs['n_counts'] = adata.X.sum(axis=1)

In [8]:
save_dir = f"/home/cavin/jt/benchmark/experiments/middle_data/geneformer/lung_cancer"

In [9]:
preprocess_path = os.path.join(save_dir, "lung_cancer_gene_tokenized.h5ad")
adata.write_h5ad(preprocess_path)

In [10]:
tk = TranscriptomeTokenizer(custom_attr_name_dict=None, 
                            nproc=16, 
                            model_version="V2")
tk.tokenize_data(save_dir, 
                save_dir, 
                "lung_cancer_gene_tokenized",
                file_format="h5ad")

Tokenizing /home/cavin/jt/benchmark/experiments/middle_data/geneformer/lung_cancer/lung_cancer_gene_tokenized.h5ad


/home/cavin/anaconda3/envs/geneformer/lib/python3.10/site-packages/geneformer/tokenizer.py:544: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  for i in adata.var["ensembl_id_collapsed"][coding_miRNA_loc]
/home/cavin/anaconda3/envs/geneformer/lib/python3.10/site-packages/geneformer/tokenizer.py:547: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  coding_miRNA_ids = adata.var["ensembl_id_collapsed"][coding_miRNA_loc]


/home/cavin/jt/benchmark/experiments/middle_data/geneformer/lung_cancer/lung_cancer_gene_tokenized.h5ad has no column attribute 'filter_pass'; tokenizing all cells.
Creating dataset.


In [11]:
@measure_resources(cuda_id=0, task_des="Geneformer run cosmx human lung cancer")
def get_emb():
    # initiate EmbExtractor
    # OF NOTE: model_version should match version of model to be used (V1 or V2) to use the correct token dictionary
    embex = EmbExtractor(model_type="CellClassifier",
                        num_classes=2,
                        emb_mode='cls',
                        max_ncells=None,
                        emb_layer=0,
                        forward_batch_size=16,
                        model_version="V2",  # OF NOTE: SET TO V1 MODEL, PROVIDE V1 MODEL PATH IN SUBSEQUENT CODE
                        nproc=4)

    # extracts embedding from input data
    # input data is tokenized rank value encodings generated by Geneformer tokenizer (see tokenizing_scRNAseq_data.ipynb)
    # example dataset for V1 model series: https://huggingface.co/datasets/ctheodoris/Genecorpus-30M/tree/main/example_input_files/cell_classification/disease_classification/human_dcm_hcm_nf.dataset
    embs = embex.extract_embs("/home/cavin/jt/benchmark/models/Geneformer-V2-104M", # example V2 fine-tuned model
                            f"{save_dir}/lung_cancer_gene_tokenized.dataset",
                            "/home/cavin/jt/benchmark/experiments/embedings/spatial_cluster_with_annotations",
                            "lung_cancer_gene_tokenized_embs")
    return embs

In [12]:
embs = get_emb()

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at /home/cavin/jt/benchmark/models/Geneformer-V2-104M and are newly initialized: ['bert.pooler.dense.bias', 'classifier.bias', 'classifier.weight', 'bert.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  0%|          | 0/5475 [00:00<?, ?it/s]


RESOURCE USAGE REPORT: get_emb
Timestamp                   : 2026-03-08 18:55:09
Target GPU ID               : 0
Execution Time (Minutes)    : 3.4480
Execution Time (Seconds)    : 206.88
CPU Peak Memory Usage (GB)  : 3.2899
GPU Peak Allocated Mem (GB) : 1.2566
GPU Peak Cached Mem (GB)    : 1.3047
CUDA Available              : True

文件已存在，数据已成功追加至末尾。
资源使用记录已保存到: /home/cavin/jt/benchmark/experiments/results/run_metric/cluster_with_annotation/record.csv


In [13]:
embs.shape

(87589, 768)

In [14]:
emb_df = pd.DataFrame(
    embs.values, 
    index=adata.obs_names,
    columns=[f"geneformer_dim_{i}" for i in range(embs.shape[1])]
)
save_path = "/home/cavin/jt/benchmark/experiments/embedings/spatial_cluster_with_annotations/geneformer_human_lung_cancer_emb.parquet"
emb_df.to_parquet(save_path, compression="gzip")

In [15]:
loaded_emb_df = pd.read_parquet(save_path)
adata = sc.read_h5ad(simple_path)
aligned_emb_df = loaded_emb_df.reindex(adata.obs_names)
adata

AnnData object with n_obs × n_vars = 87589 × 960
    obs: 'cell_type', 'niche', 'patient_sample', 'merge_cell_type', 'Area', 'AspectRatio', 'CenterX_local_px', 'CenterY_local_px', 'CenterX_global_px', 'CenterY_global_px', 'Width', 'Height', 'Mean.MembraneStain', 'Max.MembraneStain', 'Mean.PanCK', 'Max.PanCK', 'Mean.CD45', 'Max.CD45', 'Mean.CD3', 'Max.CD3', 'Mean.DAPI', 'Max.DAPI', 'fov', 'cell_ID', 'n_genes'
    var: 'n_cells'
    obsm: 'spatial'

In [16]:
adata.obsm["X_geneformer"] = aligned_emb_df.to_numpy(dtype=np.float32)
adata

AnnData object with n_obs × n_vars = 87589 × 960
    obs: 'cell_type', 'niche', 'patient_sample', 'merge_cell_type', 'Area', 'AspectRatio', 'CenterX_local_px', 'CenterY_local_px', 'CenterX_global_px', 'CenterY_global_px', 'Width', 'Height', 'Mean.MembraneStain', 'Max.MembraneStain', 'Mean.PanCK', 'Max.PanCK', 'Mean.CD45', 'Max.CD45', 'Mean.CD3', 'Max.CD3', 'Mean.DAPI', 'Max.DAPI', 'fov', 'cell_ID', 'n_genes'
    var: 'n_cells'
    obsm: 'spatial', 'X_geneformer'

In [17]:
adata.obsm["X_geneformer"]

array([[-0.53651196,  0.21571891,  0.19943973, ..., -1.8232745 ,
        -1.2567023 ,  0.36680475],
       [-0.51719505,  0.2110504 , -0.05354029, ..., -2.0593414 ,
        -1.327544  ,  0.1473998 ],
       [-0.44783267,  0.34775352, -0.12345362, ..., -1.9827502 ,
        -1.207774  ,  0.39436337],
       ...,
       [-1.0724655 , -0.514914  , -1.2409027 , ..., -2.5596852 ,
        -0.9047971 ,  0.8700316 ],
       [-0.77731943, -0.5370814 , -1.2282981 , ..., -2.7611084 ,
        -1.2142248 ,  0.318782  ],
       [-1.5676407 , -0.42285317, -1.1035324 , ..., -2.5346692 ,
        -0.97355884,  0.16322674]], dtype=float32)